In [30]:
import pandas as pd
from pprint import pprint
import json
import re
from typing import Dict, List, Any


In [31]:
ds_plannerStudy_628 = pd.read_json(
    "Original datasets/plannerStudy_628.json", lines=True
)
with open("stage2/ai_processed_tasks_names.json", "r", encoding="utf-8") as f:
    ai_processed_names = json.load(f)
with open("stage2/ai_processed_tasks_outline.json", "r", encoding="utf-8") as f:
    ai_processed_outline = json.load(f)

In [32]:
samples = []

for row in ds_plannerStudy_628.index:
    raw_output = ds_plannerStudy_628["output"][row]

    # Ensure we’re working with a dict, not a raw string
    if isinstance(raw_output, str):
        try:
            parsed_output = json.loads(raw_output)
        except json.JSONDecodeError:
            print (f"failed to parse in {row}")
            samples.append(
                {f"index_{row}": ds_plannerStudy_628["output"][row]}
            )  # Store empty dict for this sample
            continue
    else:
        parsed_output = raw_output

    # Inject timeuntiltest directly into the plan JSON
    parsed_output["timeuntiltest"] = ds_plannerStudy_628["input"][row]["timeuntiltest"]

    # Store back into samples
    samples.append({f"index_{row}": json.dumps(parsed_output)})

failed to parse in 49


In [33]:
# samples = []
# for row in ds_plannerStudy_628.index:
#     samples.append({f"index_{row}": ds_plannerStudy_628["output"][row]})

In [34]:
def reindex_samples(samples):
    """
    Reindex the dataset so each sample has a fresh sequential index.
    Input: list of dicts like [{'index_327': json_string}, ...]
    Output: list of dicts like [{'index_0': json_string}, {'index_1': json_string}, ...]
    """
    new_samples = []
    for new_idx, sample in enumerate(samples):
        # Each sample has only one key like "index_327"
        old_key = list(sample.keys())[0]
        value = sample[old_key]
        new_key = f"index_{new_idx}"
        new_samples.append({new_key: value})
    return new_samples

In [35]:
def remove_and_reindex_samples(samples, remove_indices):
    """
    Remove specified indices from the dataset and reindex sequentially.

    Args:
        samples (list): List of dicts like [{'index_327': json_string}, ...]
        remove_indices (list): List of index keys to remove, e.g. ['index_31', 'index_70', ...]

    Returns:
        list: Reindexed dataset with unwanted samples removed.
    """
    new_samples = []
    new_idx = 0

    for sample in samples:
        old_key = list(sample.keys())[0]
        if old_key in remove_indices:
            # Skip unwanted sample
            continue

        value = sample[old_key]
        new_key = f"index_{new_idx}"
        new_samples.append({new_key: value})
        new_idx += 1

    return new_samples

In [36]:
def remove_invalid_json(samples):
    """
    Remove invalid JSON samples from dataset and count them.

    Args:
        samples (list): List of dicts with JSON strings under keys like 'index_i'.

    Returns:
        tuple: (cleaned_samples, invalid_count)
    """
    cleaned_samples = []
    invalid_count = 0

    for i, sample in enumerate(samples):
        # Each sample has one key like "index_327"
        key = list(sample.keys())[0]
        raw = sample[key]

        try:
            parsed = json.loads(raw)  # Try parsing JSON
            # If valid, keep the sample
            cleaned_samples.append(sample)
        except json.JSONDecodeError:
            # If invalid, increment counter
            invalid_count += 1
            print(f"Invalid JSON at sample {i} (key={key})")
    cleaned_samples = reindex_samples(cleaned_samples)
    return cleaned_samples, invalid_count




In [37]:
# Compile all your day-name regex formats
DAYNAME_PATTERNS = [
    re.compile(
        r"^(?:Section:\s*)?(?:Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday)(?:\s*-\s*(?:Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday))?\s*\(Week\s*\d+\)(?::\s*)?(.*)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^(?:Section:\s*)?(?:Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday)\s*\(Week\s*\d+\):\s*(.*)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^(?:Section:\s*)?(?:Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday),\s*Week\s*\d+(?::\s*)?(.*)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^(?:Section:\s*)?(?:Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday)\s*-\s*Week\s*\d+(?::\s*)?(.*)$",
        re.IGNORECASE,
    ),
    re.compile(
        r"^(?:Section:\s*)?Week\s*\d+:\s*(Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday)\s*(.*)$",
        re.IGNORECASE,
    ),
]


def contains_dayname_format(section_name: str) -> bool:
    """Check if a section name matches any of the day-name formats."""
    return any(p.match(section_name) for p in DAYNAME_PATTERNS)


def remove_dayname_plans(samples):
    filtered_samples = []

    for i, sample in enumerate(samples):
        try:
            sample_key = next(iter(sample.keys()))
        except StopIteration:
            print(f"Warning: Empty sample at position {i}. Skipping.")
            continue

        raw = sample[sample_key]

        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            print(f"Warning: Sample '{sample_key}' contains invalid JSON. Skipping.")
            continue

        sections = parsed.get("sections", [])
        has_dayname_format = any(
            contains_dayname_format(sec.get("name", "")) for sec in sections
        )

        if not has_dayname_format:
            filtered_samples.append(sample)
    filtered_samples=reindex_samples(filtered_samples)
    return filtered_samples



# print(f"original length:{len(samples)}")
# filtered_samples =remove_dayname_plans(samples)
# filtered_samples=reindex_samples(filtered_samples)
# print(f"filtered length:{len(filtered_samples)}")
# # pprint(samples[0])
# print("===========")
# pprint(filtered_samples[316])

In [38]:
def summarize_sections_with_regex(
    filtered_samples, match_regex=None, exclude_regex=None, based_on="name"
):
    """
    Summarize and print plans that contain sections matching a given regex pattern.
    You can provide a match regex, an exclude regex, or both.

    Args:
        filtered_samples (list): List of dicts with JSON strings under keys like 'index_i'.
        match_regex (str, optional): Regex string to match section names.
        exclude_regex (str, optional): Regex string to exclude section names.
        based_on (str): Specify whether to match against 'name' or 'outline' of sections.
    """
    match_pattern = re.compile(match_regex, re.IGNORECASE) if match_regex else None
    exclude_pattern = (
        re.compile(exclude_regex, re.IGNORECASE) if exclude_regex else None
    )

    week_sections_summary = []
    indexing_error = 0

    for i, sample in enumerate(filtered_samples):
        key = f"index_{i}"
        try:
            raw = sample[key]
        except KeyError:
            indexing_error += 1
            print(f"Sample {i} does not have key '{key}'")
            continue

        try:
            parsed = json.loads(raw)  # convert JSON string → dict
        except json.JSONDecodeError as e:
            print(f"Error in sample {i}: {e}")
            print("Raw string:", raw[:200], "...")  # preview first 200 chars
            continue  # skip invalid JSON

        plan_name = parsed.get("name")
        sections = parsed.get("sections", [])

        # Apply match/exclude logic
        matched_sections = []
        for sec in sections:
            name = sec.get("name", "")
            outline = sec.get("outline", "")
            
            if based_on == "name":
                match_text = name
            elif based_on == "outline":
                match_text = outline
            else:
                raise ValueError("based_on must be 'name' or 'outline'")
            
            match_ok = match_pattern.search(match_text) if match_pattern else True
            exclude_ok = not (exclude_pattern and exclude_pattern.search(match_text))
            if match_ok and exclude_ok:
                matched_sections.append(sec)

        if matched_sections:
            week_sections_summary.append(
                {
                    "index": i,
                    "plan_name": plan_name,
                    "count": len(matched_sections),
                    "sections": matched_sections,
                }
            )

    # Print results
    total_matched_sections = 0
    for summary in week_sections_summary:
        print(f"Plan: {summary['plan_name']}")
        print(f"Index: {summary['index']}")
        total_matched_sections += summary["count"]
        for sec in summary["sections"]:
            print(f"\tSection: {sec['name']}")
            print(f"\tOutline: {sec['outline']}\n")

    print(f"Total matched sections found: {total_matched_sections}")
    print(f"Total plans with matching sections: {len(week_sections_summary)}")
    print(f"Indexing errors: {indexing_error}")

In [39]:
def extract_sections_by_regex(filtered_samples, regex_pattern,based_on="name"):
    """
    Filter sections by regex and output one-line JSON dataset
    with plan index, section order index, and outline.

    Args:
        filtered_samples (list): List of dicts with JSON strings under keys like 'index_i'.
        regex_pattern (str): Regex string to match section names.
        based_on (str): Specify whether to match against 'name' or 'outline' of sections.

    Returns:
        list: List of JSON strings, each one line.
    """
    compiled_pattern = re.compile(regex_pattern, re.IGNORECASE)
    results = []

    for i, sample in enumerate(filtered_samples):
        key = f"index_{i}"
        if key not in sample:
            continue

        try:
            parsed = json.loads(sample[key])
        except json.JSONDecodeError:
            continue

        sections = parsed.get("sections", [])
        for j, sec in enumerate(sections):
            name = sec.get("name", "")
            outline = sec.get("outline", "")
            
            if based_on == "name":
                match_text = name
            elif based_on == "outline":
                match_text = outline
            else:
                raise ValueError("based_on must be 'name' or 'outline'")
            
            if compiled_pattern.search(match_text):
                record = {"plan_index": i, "section_index": j, "outline": outline}
                # Save as one-line JSON string
                results.append(json.dumps(record))

    return results


# preLLM_dataset = extract_sections_by_regex(
#     filtered, r"(Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday)"
# )
# # Save as JSONL (one JSON object per line)
# with open("stage2/outlines to summarize.json", "w", encoding="utf-8") as f:
#     for record in preLLM_dataset:
#         f.write(json.dumps(record, ensure_ascii=False) + "\n")

In [40]:
DAY_PATTERN = re.compile(
    r"^(Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday):\s*(.*)$",
    re.IGNORECASE,
)

def clean_day_colon_formats(samples):
    """
    Remove day names from section titles in the dataset.
    Keeps the real title and reindexes the dataset.
    ex: "Monday: Introduction" -> "Introduction"

    Args:
        samples (list): List of dicts with JSON strings under keys like 'index_i'.

    Returns:
        list: Cleaned and reindexed dataset.
    """
    new_samples = []

    for new_idx, sample in enumerate(samples):
        old_key = list(sample.keys())[0]
        raw = sample[old_key]

        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            # Skip invalid JSON
            continue

        cleaned_sections = []
        for sec in parsed.get("sections", []):
            name = sec.get("name", "")
            match = DAY_PATTERN.match(name)
            if match:
                
                # Group 2 = real title after day name
                real_title = match.group(2).strip()
                if not real_title:
                    real_title = "Study Task"
                sec["name"] = real_title
            cleaned_sections.append(sec)

        parsed["sections"] = cleaned_sections
        new_key = f"index_{new_idx}"
        new_samples.append({new_key: json.dumps(parsed)})
    
    return new_samples



In [41]:
# Regex patterns you specified
DAY_DASH_DAY_PATTERN = re.compile(
    r"^(Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday) - Day\s*(.*)$",
    re.IGNORECASE,
)
DAY_DASH_PATTERN = re.compile(
    r"^(Sunday|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday) - \s*(.*)$",
    re.IGNORECASE,
)


def clean_day_dash_formats(samples):
    """
    Clean section names that match day-name dash formats.
    Removes the day name and dash, keeps the real title.
    Reindexes the dataset sequentially.
    ex: "Monday - Day 1: Introduction" → "Introduction"

    Args:
        samples (list): List of dicts with JSON strings under keys like 'index_i'.

    Returns:
        list: Cleaned and reindexed dataset.
    """
    new_samples = []

    for new_idx, sample in enumerate(samples):
        old_key = list(sample.keys())[0]
        raw = sample[old_key]

        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            continue

        cleaned_sections = []
        for sec in parsed.get("sections", []):
            name = sec.get("name", "")
            if DAY_DASH_DAY_PATTERN.match(name):
                real_title = DAY_DASH_DAY_PATTERN.match(name).group(2).strip()
                sec["name"] = real_title if real_title else "Study Task"
            elif DAY_DASH_PATTERN.match(name):
                real_title = DAY_DASH_PATTERN.match(name).group(2).strip()
                sec["name"] = real_title if real_title else "Study Task"
            cleaned_sections.append(sec)

        parsed["sections"] = cleaned_sections
        new_key = f"index_{new_idx}"
        new_samples.append({new_key: json.dumps(parsed)})

    return new_samples

# print(len(filtered_samples))
# day_dash_cleaned=clean_day_dash_formats(filtered_samples)
# print(len(day_dash_cleaned))

In [42]:
def apply_ai_tasks_to_dataset(original_dataset: list, ai_results: list,replace_key:str="name") -> list:
    """
    Replace section names in the original dataset with AI-processed task names.

    Args:
        original_dataset (list): List of dicts like {'index_599': '...json string...'}
        ai_results (list): List of dicts like {"plan_index": 599, "section_index": 6, "task": "..."}
        replace_key (str): The key in the section dict to replace with the AI task (e.g., "name" or "outline")

    Returns:
        list: Updated dataset with replaced section names.
    """
    # Build a lookup for faster access
    ai_lookup = {}
    for task in ai_results:
        ai_lookup.setdefault(task["plan_index"], []).append(task)

    updated_dataset = []
    for sample in original_dataset:
        key = list(sample.keys())[0]  # e.g. "index_599"
        plan_index = int(key.split("_")[1])

        parsed = json.loads(sample[key])

        if plan_index in ai_lookup:
            for task in ai_lookup[plan_index]:
                section_idx = task["section_index"]
                if section_idx < len(parsed["sections"]):
                    parsed["sections"][section_idx][replace_key] = task["task"]

        # Save back
        updated_dataset.append({key: json.dumps(parsed, ensure_ascii=False)})

    return updated_dataset

In [43]:
def deduplicate_dataset(input_file: str, output_file: str) -> None:
    """
    Remove duplicate rows from a dataset, keeping the first occurrence.

    Args:
        input_file (str): Path to the checkpoint JSON file.
        output_file (str): Path to save the deduplicated dataset.
    """
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)  # list of dicts

    seen = set()
    deduped = []
    for row in data:
        # Use (plan_index, section_index) as unique key
        key = (row["plan_index"], row["section_index"])
        if key not in seen:
            seen.add(key)
            deduped.append(row)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(deduped, f, indent=2, ensure_ascii=False)

    print(f"✅ Deduplicated dataset saved to {output_file}")
    print(f"Original size: {len(data)}, Deduplicated size: {len(deduped)}")

In [44]:
def normalize_time_until_test(samples):
    """
    Normalize 'timeuntiltest' values into integer days.
    Handles formats like '6 day(s)', '1 week(s)', etc.

    Args:
        samples (list): List of dicts with JSON strings under keys like 'index_i'.

    Returns:
        list: Dataset with normalized 'timeuntiltest' values as integers.
    """
    new_samples = []
    DAY_PATTERN = re.compile(r"(\d+)\s*day\(s\)", re.IGNORECASE)
    WEEK_PATTERN = re.compile(r"(\d+)\s*week\(s\)", re.IGNORECASE)

    for new_idx, sample in enumerate(samples):
        old_key = list(sample.keys())[0]
        raw = sample[old_key]

        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            # Skip invalid JSON
            continue

        time_str = parsed.get("timeuntiltest", "").strip()
        normalized_days = None

        if match := DAY_PATTERN.match(time_str):
            normalized_days = int(match.group(1))
        elif match := WEEK_PATTERN.match(time_str):
            normalized_days = int(match.group(1)) * 7

        # Default fallback if parsing fails
        if normalized_days is None:
            normalized_days = 0

        parsed["timeuntiltest"] = normalized_days

        new_key = f"index_{new_idx}"
        new_samples.append({new_key: json.dumps(parsed)})

    return new_samples

In [45]:
deduplicate_dataset(
    "stage2/ai_processed_tasks_names.json", "stage2/ai_processed_tasks_names.json"
)

✅ Deduplicated dataset saved to stage2/ai_processed_tasks_names.json
Original size: 1855, Deduplicated size: 1855


In [46]:
filtered=remove_dayname_plans(samples)
filtered=clean_day_colon_formats(filtered)
filtered=clean_day_dash_formats(filtered)
filtered = apply_ai_tasks_to_dataset(filtered, ai_processed_names,replace_key="name")
filtered = apply_ai_tasks_to_dataset(filtered, ai_processed_outline, replace_key="outline")
filtered = normalize_time_until_test(filtered)
len(filtered)

602

In [47]:
# for i, sample in enumerate(filtered):
#     key = f"index_{i}"
#     try:
#         raw = sample[key]
#     except KeyError:
#         indexing_error += 1
#         print(f"Sample {i} does not have key '{key}'")
#         continue

#     try:
#         parsed = json.loads(raw)  # convert JSON string → dict
#     except json.JSONDecodeError as e:
#         print(f"Error in sample {i}: {e}")
#         print("Raw string:", raw[:200], "...")  # preview first 200 chars
#         continue  # skip invalid JSON

#     # plan_name = parsed.get("name")
#     time_horizon = parsed.get("timeuntiltest")
#     print(f"index_{i},\n\t Time until test: {time_horizon}")

In [48]:
preLLM_dataset = extract_sections_by_regex(filtered, r"^\d+$", based_on="name")
# Save as JSONL (one JSON object per line)
with open("stage2/outlines to summarize.json", "w", encoding="utf-8") as f:
    for record in preLLM_dataset:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

In [49]:
def expand_week_sections(plan: dict) -> dict:
    """
    Replace sections with names like 'Week 1: ...' into 5 daily tasks + 2 break days.
    Returns a new plan dict with expanded sections.
    """
    new_sections = []
    try:
        plan = json.loads(plan)
    except json.JSONDecodeError:
        print("Failed to parse plan as JSON")
        return plan  # Return original plan if parsing fails
    # print(plan.values())
    # plan=plan.values()
    for sec in plan.get("sections", []):
        try:
            name = sec.get("name", "")
            # print(name)
        except Exception as e:
            print(
                f"Error processing section in plan {plan.get('name', 'Unknown')}: {e}"
            )
            continue
        match = re.match(r"(Week\s*\d+):\s*(.*)", name, flags=re.IGNORECASE)
        if match:
            week_label, base_title = match.groups()
            # Create 5 study days
            for i in range(1, 6):
                new_sections.append(
                    {
                        "index": i - 1,
                        "task": f"{base_title} - Day {i}",
                        "description": sec.get("outline", ""),
                        "is_repeatable": False,
                        "repeat_frequency": 0,
                        "estimated_duration_minutes": {"min": 30, "max": 45},
                        "gap_flag": False,
                    }
                )
            # Add 2 break days
            for i in range(1, 3):
                new_sections.append(
                    {
                        "index": i - 1,
                        "task": f"Break Day",
                        "description": "Take a rest day to recharge and consolidate learning.",
                        "is_repeatable": False,
                        "repeat_frequency": 0,
                        "estimated_duration_minutes": {"min": 0, "max": 0},
                        "gap_flag": True,
                    }
                )
        else:
            # Keep non-week sections unchanged
            new_sections.append(sec)

    # Return updated plan
    return {**plan, "sections": new_sections}


# s = expand_week_sections(samples[624]["index_624"])
# pprint([sec.get("task") for sec in s.get("sections", [])])
# pprint(s)

In [50]:
def expand_day_range_sections(plan: str) -> dict:
    """
    Expand sections with names like 'Day 1-3: Title' into multiple daily tasks.
    Returns a new plan dict with expanded sections.
    """
    new_sections = []
    try:
        plan = json.loads(plan)
    except json.JSONDecodeError:
        print("Failed to parse plan as JSON")
        return plan  # Return original plan if parsing fails

    for sec in plan.get("sections", []):
        name = sec.get("name", "")
        match = re.match(r"^Day\s+(\d+)-(\d+):\s*(.+)$", name, flags=re.IGNORECASE)
        if match:
            start_day, end_day, base_title = match.groups()
            start_day, end_day = int(start_day), int(end_day)

            # Create tasks for each day in the range
            for day in range(start_day, end_day + 1):
                new_sections.append(
                    {
                        "index": day - 1,
                        "task": f"Day {day}: {base_title}",
                        "description": sec.get("outline", ""),
                        "is_repeatable": False,
                        "repeat_frequency": 0,
                        "estimated_duration_minutes": {"min": 30, "max": 45},
                        "gap_flag": False,
                    }
                )
        else:
            # Keep non-day-range sections unchanged
            new_sections.append(sec)

    # Return updated plan
    return {**plan, "sections": new_sections}

In [51]:
def find_week1_duplicates(samples):
    """
    Scan a list-like structure of study plan samples and return indexes
    where 'Week 1' appears in more than one section name.

    Args:
        samples (list or dict-like): Each entry is a dict with a single key
                                     like 'index_78' and a JSON string as value.

    Returns:
        dict: {"count": int, "indexes": [list of indexes]}
    """
    result_indexes = []

    for sample in samples:
        # Each sample is a dict with one key (e.g., 'index_78')
        for idx, plan_json in sample.items():
            plan = json.loads(plan_json)
            sections = plan.get("sections", [])

            # Count how many section names contain "Week 1"
            week1_count = sum(1 for s in sections if "Week 1" in s.get("name", ""))

            if week1_count > 1:
                result_indexes.append(idx)

    return {"count": len(result_indexes), "indexes": result_indexes}


In [52]:
remove_list = [
    "index_31",
    "index_70",
    "index_71",
    "index_78",
    "index_93",
    "index_149",
    "index_215",
    "index_246",
    "index_249",
    "index_256",
    "index_364",
    "index_516",
]
samples = remove_and_reindex_samples(samples, remove_list)
samples, invalid_count = remove_invalid_json(samples)

def preprocess_dataset(samples, ai_processed_names, ai_processed_outline):
    """
    Apply all preprocessing steps and convert to target schema.
    """

    # Step 1: Remove invalid JSON
    filtered, invalid_count = remove_invalid_json(samples)
    print(f"Removed {invalid_count} invalid rows")

    # Step 2: Normalize day-based section names
    filtered = remove_dayname_plans(filtered)
    filtered = clean_day_colon_formats(filtered)
    filtered = clean_day_dash_formats(filtered)

    # Step 3: Apply AI-processed replacements
    filtered = apply_ai_tasks_to_dataset(
        filtered, ai_processed_names, replace_key="name"
    )
    filtered = apply_ai_tasks_to_dataset(
        filtered, ai_processed_outline, replace_key="outline"
    )
    filtered = normalize_time_until_test(filtered)

    # Step 4: Convert to target schema
    converted = []
    for i, sample in enumerate(filtered):
        key = list(sample.keys())[0]
        plan = json.loads(sample[key])

        # --- Apply expansions here ---
        plan = expand_day_range_sections(json.dumps(plan))
        plan = expand_week_sections(json.dumps(plan))

        # Build schema
        schema = {
            "goal": plan.get("name", ""),
            "goal_category": "one_time",  # or "habit" depending on your logic
            "goal_type": "study",  # you can customize
            "time_horizon": plan.get("timeuntiltest", ""),
            "description": plan.get("description", ""),
            "baseline": {
                "fixed_constraints": "",
                "physical_constraints": "",
                "non_negotiables": "",
            },
            "tasks": [],
        }

        # Convert sections into tasks
        for j, sec in enumerate(plan.get("sections", [])):
            schema["tasks"].append(
                {
                    "index": j,
                    "task": sec.get("name", "") or sec.get("task", ""),
                    "description": sec.get("outline", "") or sec.get("description", ""),
                    "is_repeatable": sec.get("is_repeatable", False),
                    "repeat_frequency": sec.get("repeat_frequency", 0),
                    "gap_flag": sec.get("gap_flag", False),
                    "estimated_duration_minutes": sec.get(
                        "estimated_duration_minutes", {"min": 30, "max": 45}
                    ),
                }
            )

        converted.append(schema)

    return converted

Invalid JSON at sample 48 (key=index_48)


In [53]:
converted_dataset = preprocess_dataset(samples, ai_processed_names, ai_processed_outline)   


Removed 0 invalid rows


In [54]:
def save_for_finetune(converted, filename="converted_dataset.jsonl"):
    """
    Save the converted dataset in JSONL format for LLM fine-tuning.

    Args:
        converted (list): List of schema dicts produced by preprocess_dataset.
        filename (str): Output file name.
    """
    with open(filename, "w", encoding="utf-8") as f:
        for sample in converted:
            # Each sample becomes one JSON line
            f.write(json.dumps(sample, ensure_ascii=False) + "\n")

In [55]:
save_for_finetune(converted_dataset, "final/plannerStudy_converted_dataset.jsonl")

In [56]:
converted_dataset[1]

{'goal': 'Ecological Impacts of Microplastic Pollution on Marine Food Webs Study Plan',
 'goal_category': 'one_time',
 'goal_type': 'study',
 'time_horizon': 21,
 'description': 'This study plan is designed to help you prepare for your test on the ecological impacts of microplastic pollution on marine food webs. The plan is structured to fit your fast learning pace and low effort level, ensuring you cover all necessary topics efficiently over the next three weeks.',
 'baseline': {'fixed_constraints': '',
  'physical_constraints': '',
  'non_negotiables': ''},
 'tasks': [{'index': 0,
   'task': 'Introduction and Basic Concepts - Day 1',
   'description': 'Start by familiarizing yourself with the basic concepts of microplastics and their sources. Use the Pomodoro technique to break your study sessions into manageable chunks. Spend 25 minutes watching introductory videos, followed by a 5-minute break. Repeat this cycle for 2 hours each day. Focus on understanding what microplastics are, t

In [57]:
# test=[]
# with open("final/plannerStudy_converted_dataset.jsonl", "r", encoding="utf-8") as f:
#     for line in f:
#         record = json.loads(line)  # Parse each line as JSON
#         test.append(record)